<a href="https://colab.research.google.com/github/msdurk/masters/blob/main/machinewars_vs_ephish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas numpy

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving machinewars_emails_entity_rephrased_v1.json to machinewars_emails_entity_rephrased_v1.json


In [ ]:
train_val_paths = [
    "/content/data_part_1.json",
    "/content/data_part_2.json",
    "/content/data_part_3.json",
    "/content/data_part_4.json",
]

ephish_path = "/content/ephish_en_subject_body_label.jsonl"

In [ ]:
import json
import pandas as pd

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data)

def load_jsonl_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def normalize_label_from_type(x):
    """
    For data_part_*.json:
    phishing -> 1
    everything else -> 0
    """
    x = str(x).strip().lower()
    return 1 if x == "phishing" else 0

def build_dataframe_from_parts(paths):
    """
    For data_part_*.json files with columns like Subject, Body, Type
    """
    frames = []

    for path in paths:
        df = load_json_file(path)

        for col in ["Subject", "Body", "Type"]:
            if col not in df.columns:
                df[col] = ""

        df["Subject"] = df["Subject"].fillna("").astype(str)
        df["Body"] = df["Body"].fillna("").astype(str)
        df["Type"] = df["Type"].fillna("").astype(str)

        df["text"] = (
            "Subject: " + df["Subject"].str.strip() +
            "\n\nBody: " + df["Body"].str.strip()
        )
        df["label"] = df["Type"].apply(normalize_label_from_type)

        frames.append(df[["text", "label"]])

    full_df = pd.concat(frames, ignore_index=True)
    full_df = full_df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return full_df

def build_dataframe_from_jsonl(path):
    """
    For ephish_en_subject_body_label.jsonl
    Expected columns:
      Subject, Body, label
    """
    df = load_jsonl_file(path)

    for col in ["Subject", "Body", "label"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {path}")

    df["Subject"] = df["Subject"].fillna("").astype(str)
    df["Body"] = df["Body"].fillna("").astype(str)
    df["label"] = df["label"].astype(int)

    df["text"] = (
        "Subject: " + df["Subject"].str.strip() +
        "\n\nBody: " + df["Body"].str.strip()
    )

    df = df[["text", "label"]].dropna(subset=["text", "label"]).reset_index(drop=True)
    return df

In [ ]:
train_val_df = build_dataframe_from_parts(train_val_paths)
test_df = build_dataframe_from_jsonl(ephish_path)

print("Train/Val shape:", train_val_df.shape)
print("Ephish test shape:", test_df.shape)

print("\nTrain/Val labels:")
print(train_val_df["label"].value_counts())

print("\nEphish labels:")
print(test_df["label"].value_counts())

train_val_df.head()

Train/Val shape: (19800, 2)
Ephish test shape: (11502, 2)

Train/Val labels:
label
0    13044
1     6756
Name: count, dtype: int64

Ephish labels:
label
1    5996
0    5506
Name: count, dtype: int64


,text,label
0,Subject: Unusual Sign-in Activity Detected on ...,1
1,Subject: Security Alert: New Sign-in to Your G...,1
2,Subject: Important Security Notification Regar...,1
3,Subject: Unusual Login Activity Detected on Yo...,1
4,Subject: Security Alert: New Sign-In to Your S...,1


In [ ]:
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_val_df["label"]
)

In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Enable faster matmul on Ampere/A100
torch.set_float32_matmul_precision("high")

# Optional: allow TF32 on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(device)

cuda


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(labels, preds)

    try:
        roc_auc = roc_auc_score(labels, probs[:, 1])
    except:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/bert_phishing_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.045609,0.092681,0.977525,0.983155,0.950407,0.966504,0.997431
2,0.027183,0.043805,0.990152,0.980234,0.991118,0.985646,0.999272
3,0.021432,0.041770,0.991162,0.983824,0.990377,0.987090,0.999386


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2970, training_loss=0.050308607749465335, metrics={'train_runtime': 152.2919, 'train_samples_per_second': 312.032, 'train_steps_per_second': 19.502, 'total_flos': 6291601193259840.0, 'train_loss': 0.050308607749465335, 'epoch': 3.0})

In [ ]:
try:
    model = torch.compile(model)
    print("Model compiled successfully.")
except Exception as e:
    print("torch.compile not available or failed:", e)

Model compiled successfully.


In [ ]:
print("=== Validation (in-domain) ===")
val_metrics = trainer.evaluate()
print(val_metrics)

=== Validation (in-domain) ===


{'eval_loss': 0.042487502098083496, 'eval_accuracy': 0.9916666666666667, 'eval_precision': 0.9831378299120235, 'eval_recall': 0.9925980754996299, 'eval_f1': 0.9878453038674033, 'eval_roc_auc': 0.9993818017061592, 'eval_runtime': 4.7355, 'eval_samples_per_second': 836.232, 'eval_steps_per_second': 26.185, 'epoch': 3.0}


In [ ]:
print("\n=== Ephish Test (out-of-domain) ===")

test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
test_ds = test_ds.map(tokenize_function, batched=True)

test_metrics = trainer.evaluate(test_ds)
print(test_metrics)


=== Ephish Test (out-of-domain) ===


Map:   0%|          | 0/11502 [00:00<?, ? examples/s]

{'eval_loss': 0.8774489760398865, 'eval_accuracy': 0.7951660580768563, 'eval_precision': 0.8176265270506108, 'eval_recall': 0.7813542361574383, 'eval_f1': 0.7990789698106772, 'eval_roc_auc': 0.8670254379539138, 'eval_runtime': 6.9654, 'eval_samples_per_second': 1651.294, 'eval_steps_per_second': 51.684, 'epoch': 3.0}


In [ ]:
save_dir = "/content/final_phishing_bert"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/final_phishing_bert


In [ ]:
def predict_email(subject, body):
    text = f"Subject: {subject}\n\nBody: {body}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))

    return {
        "predicted_label": pred,   # 1 = phishing, 0 = not phishing
        "phishing_prob": float(probs[1]),
        "not_phishing_prob": float(probs[0]),
    }

example = predict_email(
    subject="Security alert: unusual login attempt",
    body="We detected suspicious activity on your account. Please verify immediately."
)

example

{'predicted_label': 1,
 'phishing_prob': 0.9996880292892456,
 'not_phishing_prob': 0.00031202001264318824}

In [ ]:
def perturb_char_noise(text, p=0.03):
    swaps = {"o": "0", "i": "1", "e": "3", "a": "@", "s": "$"}
    chars = list(text)

    for i, ch in enumerate(chars):
        if ch.lower() in swaps and random.random() < p:
            chars[i] = swaps[ch.lower()]

    return "".join(chars)

def perturb_delete_keywords(text, keywords=None):
    if keywords is None:
        keywords = ["verify", "security", "urgent", "account", "password", "login"]

    out = text
    for kw in keywords:
        out = re.sub(rf"\b{re.escape(kw)}\b", "", out, flags=re.IGNORECASE)

    out = re.sub(r"\s+", " ", out).strip()
    return out

def perturb_truncate(text, max_words=50):
    return " ".join(text.split()[:max_words])

def perturb_url_mask(text):
    return re.sub(r"https?://\S+|www\.\S+", "[LINK]", text)

def perturb_subject_only(text):
    m = re.search(r"Subject:\s*(.*?)\n\s*\nBody:", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

def perturb_body_only(text):
    m = re.search(r"Body:\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

In [ ]:
def batch_predict(texts):
    preds = []
    probs_out = []

    model.eval()

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

        preds.append(int(np.argmax(probs)))
        probs_out.append(float(probs[1]))

    return preds, probs_out

In [ ]:
def batch_predict_fast(texts, model=model, tokenizer=tokenizer, batch_size=64, max_length=512):
    preds = []
    probs_out = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        preds.extend(np.argmax(probs, axis=1).tolist())
        probs_out.extend(probs[:, 1].tolist())

    return preds, probs_out

In [ ]:
def evaluate_under_attacks(df_eval):
    attacks = {
        "clean": lambda x: x,
        "char_noise": lambda x: perturb_char_noise(x, p=0.03),
        "delete_keywords": perturb_delete_keywords,
        "truncate_50w": lambda x: perturb_truncate(x, max_words=50),
        "url_mask": perturb_url_mask,
        "subject_only": perturb_subject_only,
        "body_only": perturb_body_only,
    }

    results = []

    y_true = df_eval["label"].tolist()

    for attack_name, attack_fn in attacks.items():
        attacked_texts = df_eval["text"].apply(attack_fn).tolist()
        y_pred, y_prob = batch_predict_fast(attacked_texts)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )
        acc = accuracy_score(y_true, y_pred)

        try:
            roc_auc = roc_auc_score(y_true, y_prob)
        except:
            roc_auc = float("nan")

        results.append({
            "attack": attack_name,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc,
        })

    return pd.DataFrame(results)

robustness_results = evaluate_under_attacks(val_df.reset_index(drop=True))
robustness_results.sort_values("f1", ascending=False)

,attack,accuracy,precision,recall,f1,roc_auc
0,clean,0.991162,0.983824,0.990377,0.987090,0.999387
6,body_only,0.990657,0.985947,0.986677,0.986312,0.999397
1,char_noise,0.989646,0.984467,0.985196,0.984832,0.999368
2,delete_keywords,0.989394,0.984456,0.984456,0.984456,0.999120
4,url_mask,0.981566,0.967789,0.978534,0.973132,0.998028
3,truncate_50w,0.940404,0.868961,0.971873,0.917540,0.990928
5,subject_only,0.887626,0.771257,0.953368,0.852698,0.971060


In [ ]:
candidate_models = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base",
]

summary_results = []

for model_name in candidate_models:
    print(f"\n===== Training {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    train_ds_tmp = Dataset.from_pandas(train_df.reset_index(drop=True))
    val_ds_tmp = Dataset.from_pandas(val_df.reset_index(drop=True))

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)

    train_ds_tmp = train_ds_tmp.map(tok, batched=True)
    val_ds_tmp = val_ds_tmp.map(tok, batched=True)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=f"/content/{model_name.replace('/', '_')}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds_tmp,
        eval_dataset=val_ds_tmp,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    summary_results.append({
        "model": model_name,
        "accuracy": metrics.get("eval_accuracy"),
        "precision": metrics.get("eval_precision"),
        "recall": metrics.get("eval_recall"),
        "f1": metrics.get("eval_f1"),
        "roc_auc": metrics.get("eval_roc_auc"),
    })

results_df = pd.DataFrame(summary_results)
results_df.sort_values("f1", ascending=False)


===== Training distilbert-base-uncased =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.054827,0.044584,0.988384,0.980132,0.985936,0.983026,0.998930
2,0.008085,0.038890,0.991414,0.990320,0.984456,0.987379,0.999548


,model,accuracy,precision,recall,f1,roc_auc
2,roberta-base,0.991414,0.990320,0.984456,0.987379,0.999548
1,bert-base-uncased,0.990152,0.981645,0.989637,0.985625,0.999446
0,distilbert-base-uncased,0.988131,0.983680,0.981495,0.982586,0.999189


In [ ]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import torch

from nltk.corpus import wordnet
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

In [ ]:
def predict_one(text, model, tokenizer):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))
    return pred, float(probs[1])

In [ ]:

def predict_one_fast(text, model, tokenizer, max_length=512):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

    probs = probs.detach().float().cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1])

In [ ]:
def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)

def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = text.split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)
        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]
            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [ ]:
def get_token_saliency(text, model, tokenizer):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    embedding_layer = model.get_input_embeddings()
    inputs_embeds = embedding_layer(input_ids).detach()
    inputs_embeds.requires_grad_(True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    pred_class = torch.argmax(outputs.logits, dim=1)
    score = outputs.logits[0, pred_class]
    score.backward()

    grads = inputs_embeds.grad[0]                  # [seq_len, hidden]
    saliency = grads.norm(dim=1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    token_scores = []
    for tok, s in zip(tokens, saliency):
        token_scores.append((tok, float(s)))

    return token_scores

In [ ]:
def merge_wordpiece_tokens(token_scores):
    words = []
    current_word = ""
    current_score = 0.0

    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}

    for tok, score in token_scores:
        if tok in special_tokens:
            continue

        # BERT wordpiece
        if tok.startswith("##"):
            current_word += tok[2:]
            current_score += score

        # RoBERTa space-prefix token
        elif tok.startswith("Ġ"):
            if current_word:
                words.append((current_word, current_score))
            current_word = tok[1:]
            current_score = score

        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    cleaned = []
    for w, s in words:
        w = w.strip()
        w = re.sub(r"[^\w@.\-:/]", "", w)
        if w:
            cleaned.append((w, s))

    return cleaned

In [ ]:
def important_word_deletion_attack(text, model, tokenizer, k=5):
    token_scores = get_token_saliency(text, model, tokenizer)
    word_scores = merge_wordpiece_tokens(token_scores)

    ranked_words = sorted(word_scores, key=lambda x: x[1], reverse=True)

    stop = {
        "subject", "body", "the", "a", "an", "and", "or", "to", "of", "in",
        "for", "on", "at", "is", "are", "this", "that", "with", "from"
    }

    targets = []
    for w, _ in ranked_words:
        wl = w.lower()
        if len(wl) >= 3 and wl not in stop and wl.isprintable():
            targets.append(wl)
        if len(targets) >= k:
            break

    attacked_words = []
    for w in text.split():
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in targets:
            continue
        attacked_words.append(w)

    return " ".join(attacked_words)

In [ ]:
def iterative_deletion_attack(text, model, tokenizer, max_steps=10):
    original_pred, original_prob = predict_one(text, model, tokenizer)
    current_text = text

    for step in range(1, max_steps + 1):
        new_text = important_word_deletion_attack(
            current_text, model, tokenizer, k=1
        )

        # no further change possible
        if new_text == current_text:
            break

        new_pred, new_prob = predict_one(new_text, model, tokenizer)

        if new_pred != original_pred:
            return {
                "attacked_text": new_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
            }

        current_text = new_text

    final_pred, final_prob = predict_one(current_text, model, tokenizer)
    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
    }

In [ ]:
def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + text

In [ ]:
def evaluate_attack(df_eval, attack_name, attack_fn, model, tokenizer):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].tolist()]
    y_true = df_eval["label"].tolist()
    y_pred, y_prob = batch_predict(attacked_texts)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except:
        roc_auc = float("nan")

    return {
        "attack": attack_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [ ]:
def evaluate_iterative_attack(df_eval, model, tokenizer, max_samples=None, max_steps=10):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []
    flips = 0
    steps_used = []

    attacked_examples = []

    for _, row in df_local.iterrows():
        text = row["text"]
        label = int(row["label"])

        result = iterative_deletion_attack(
            text, model, tokenizer, max_steps=max_steps
        )

        final_pred, final_prob = predict_one(result["attacked_text"], model, tokenizer)

        y_true.append(label)
        y_pred_after.append(final_pred)
        y_prob_after.append(final_prob)
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        attacked_examples.append({
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_after, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_deletion_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
    }

    attacked_examples_df = pd.DataFrame(attacked_examples)
    return summary, attacked_examples_df

In [ ]:
attack_results_val = []

attack_results_val.append(
    evaluate_attack(val_df, "clean", lambda x: x, model, tokenizer)
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "important_word_delete_k5",
        lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "prefix_injection",
        prefix_injection_attack,
        model,
        tokenizer,
    )
)

iter_summary_val, iter_examples_val = evaluate_iterative_attack(
    val_df,
    model,
    tokenizer,
    max_samples=300,   # increase if you want
    max_steps=8
)
attack_results_val.append(iter_summary_val)

attack_results_val_df = pd.DataFrame(attack_results_val)
attack_results_val_df.sort_values("f1", ascending=False)

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used
0,clean,0.991162,0.983113,0.991118,0.987099,0.999357,NaN,NaN
3,prefix_injection,0.990152,0.980938,0.990377,0.985635,0.999122,NaN,NaN
4,iterative_deletion_8steps,0.990000,0.980392,0.990099,0.985222,0.999055,0.006667,7.96
1,synonym_attack,0.989646,0.985185,0.984456,0.984820,0.999139,NaN,NaN
2,important_word_delete_k5,0.989394,0.983026,0.985936,0.984479,0.998499,NaN,NaN


In [ ]:
attack_results_test = []

attack_results_test.append(
    evaluate_attack(test_df, "clean", lambda x: x, model, tokenizer)
)

attack_results_test.append(
    evaluate_attack(
        test_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        model,
        tokenizer,
    )
)

attack_results_test.append(
    evaluate_attack(
        test_df,
        "important_word_delete_k5",
        lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
        model,
        tokenizer,
    )
)

attack_results_test.append(
    evaluate_attack(
        test_df,
        "prefix_injection",
        prefix_injection_attack,
        model,
        tokenizer,
    )
)

iter_summary_test, iter_examples_test = evaluate_iterative_attack(
    test_df,
    model,
    tokenizer,
    max_steps=8
)
attack_results_test.append(iter_summary_test)

attack_results_test_df = pd.DataFrame(attack_results_test)
attack_results_test_df.sort_values("f1", ascending=False)

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used
2,important_word_delete_k5,0.800296,0.844478,0.756171,0.797888,0.861876,NaN,NaN
0,clean,0.786124,0.794667,0.795197,0.794932,0.862067,NaN,NaN
3,prefix_injection,0.767345,0.753900,0.822048,0.786501,0.863007,NaN,NaN
4,iterative_deletion_8steps,0.775430,0.809991,0.743662,0.775411,0.858163,0.225961,6.854895
1,synonym_attack,0.776995,0.825955,0.724983,0.772182,0.835095,NaN,NaN


In [ ]:
attack_results_val = []

attack_results_val.append(
    evaluate_attack(val_df, "clean", lambda x: x, model, tokenizer)
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "important_word_delete_k5",
        lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "prefix_injection",
        prefix_injection_attack,
        model,
        tokenizer,
    )
)

iter_summary_val, iter_examples_val = evaluate_iterative_attack(
    val_df,
    model,
    tokenizer,
    max_steps=8
)
attack_results_val.append(iter_summary_val)

attack_results_val_df = pd.DataFrame(attack_results_val)
attack_results_val_df.sort_values("f1", ascending=False)

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used
0,clean,0.991162,0.983824,0.990377,0.987090,0.999387,NaN,NaN
1,synonym_attack,0.990657,0.985947,0.986677,0.986312,0.999296,NaN,NaN
3,prefix_injection,0.988889,0.977356,0.990377,0.983824,0.999299,NaN,NaN
2,important_word_delete_k5,0.988384,0.978022,0.988157,0.983063,0.998808,NaN,NaN
4,iterative_deletion_8steps,0.986364,0.975092,0.985196,0.980118,0.998757,0.010859,7.942677


In [ ]:
attack_results_val_df["dataset"] = "validation"
attack_results_test_df["dataset"] = "test"

combined_attack_results = pd.concat(
    [attack_results_val_df, attack_results_test_df],
    ignore_index=True
)

combined_attack_results = combined_attack_results[
    [
        "dataset", "attack", "accuracy", "precision", "recall",
        "f1", "roc_auc"
    ] + [c for c in combined_attack_results.columns if c in ["attack_success_rate", "avg_steps_used"]]
]

combined_attack_results.sort_values(["dataset", "f1"], ascending=[True, False])

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used
7,test,important_word_delete_k5,0.800296,0.844478,0.756171,0.797888,0.861876,NaN,NaN
5,test,clean,0.786124,0.794667,0.795197,0.794932,0.862067,NaN,NaN
8,test,prefix_injection,0.767345,0.753900,0.822048,0.786501,0.863007,NaN,NaN
9,test,iterative_deletion_8steps,0.775430,0.809991,0.743662,0.775411,0.858163,0.225961,6.854895
6,test,synonym_attack,0.776995,0.825955,0.724983,0.772182,0.835095,NaN,NaN
0,validation,clean,0.991162,0.983824,0.990377,0.987090,0.999387,NaN,NaN
1,validation,synonym_attack,0.990657,0.985947,0.986677,0.986312,0.999296,NaN,NaN
3,validation,prefix_injection,0.988889,0.977356,0.990377,0.983824,0.999299,NaN,NaN
2,validation,important_word_delete_k5,0.988384,0.978022,0.988157,0.983063,0.998808,NaN,NaN
4,validation,iterative_deletion_8steps,0.986364,0.975092,0.985196,0.980118,0.998757,0.010859,7.942677


In [ ]:
iter_examples_val[iter_examples_val["flipped"] == True].head(10)

,original_text,attacked_text,original_pred,new_pred,flipped,steps
178,Subject: Your Special Invitation to the Summit...,Subject: Your Special Invitation to the Summit...,0,1,True,2
211,Subject: Security Notification: Amazon Account...,Subject: Security Notification: Amazon Login f...,1,0,True,2


In [ ]:
iter_examples_test[iter_examples_test["flipped"] == True].head(10)

,original_text,attacked_text,original_pred,new_pred,flipped,steps
2,Subject: Exploring Collaboration Opportunities...,Subject: Exploring Collaboration Opportunities...,0,1,True,1
9,Subject: Internal Meeting: Online Accessories ...,Subject: Internal Meeting: Online Launch Strat...,0,1,True,4
12,Subject: Company Performance Update and Future...,Subject: Performance Update and Future Goals B...,1,0,True,3
13,Subject: Logistics Inquiry for Upcoming Trade ...,Subject: Inquiry for Upcoming Show Body: Dear ...,0,1,True,8
16,Subject: Inquiry: Fine Wines for Gala Event in...,Subject: Fine for Gala Event in New York Body:...,0,1,True,7
21,Subject: Request for Information on New Eco-Fr...,Subject: Request for Information on New Eco-Fr...,0,1,True,1
26,Subject: Proposta di incontro per esplorare nu...,Subject: Proposta di incontro per esplorare nu...,0,1,True,5
30,Subject: Request for Technical Feasibility Stu...,Subject: for Technical Feasibility Study on So...,1,0,True,2
35,Subject: Proposal for Organic Product Partners...,Subject: Proposal for Organic Product Partners...,0,1,True,4
37,Subject: Inquiry About Shipping Options and Pa...,Subject: Inquiry About Shipping Options and Pa...,0,1,True,5


In [ ]:
sample_text = val_df.iloc[0]["text"]

token_scores = get_token_saliency(sample_text, model, tokenizer)
word_scores = merge_wordpiece_tokens(token_scores)

pd.DataFrame(
    sorted(word_scores, key=lambda x: x[1], reverse=True)[:20],
    columns=["word", "saliency"]
)

,word,saliency
0,mail,9.971901
1,-,7.985861
2,e,4.798193
3,documentos,2.749177
4,documentos,2.608819
5,para,2.527359
6,seu,2.161697
7,uma,2.052487
8,digital,1.507084
9,tenham,1.268662


In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def benign_prefix_attack(text):
    prefix = (
        "This is a routine internal business communication from the IT department. "
        "This message is safe, legitimate, and part of standard company operations.\n\n"
    )
    return prefix + text

def benign_suffix_attack(text):
    suffix = (
        "\n\nThis is a routine internal message. "
        "Please contact the internal IT help desk if you have any questions."
    )
    return text + suffix

def contradiction_attack(text):
    prefix = (
        "No action is required at this time. "
        "This message is only informational and does not require account verification.\n\n"
    )
    return prefix + text

def training_context_attack(text):
    prefix = (
        "This is a simulated phishing-awareness training email used for internal education. "
        "It is safe and authorized.\n\n"
    )
    return prefix + text

def noise_injection_attack(text, repeat=3):
    noise = (
        "This is a general update regarding internal documentation, procedures, "
        "and routine operational processes. "
    )
    return (noise * repeat) + text

In [ ]:
def iterative_addition_attack(
    text,
    model,
    tokenizer,
    max_steps=5,
    strategy="random",
    seed=42,
):
    """
    Iteratively adds benign / contradictory / contextual text until the model flips.

    Returns a dictionary with:
      - attacked_text
      - flipped
      - steps
      - original_pred
      - new_pred
      - original_prob
      - new_prob
      - history
    """

    rng = random.Random(seed)

    attack_functions = {
        "benign_prefix": benign_prefix_attack,
        "benign_suffix": benign_suffix_attack,
        "contradiction": contradiction_attack,
        "training_context": training_context_attack,
        "noise_injection": noise_injection_attack,
    }

    ordered_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        if strategy == "random":
            attack_name, attack_fn = rng.choice(ordered_fns)

        elif strategy == "fixed":
            attack_name, attack_fn = ordered_fns[(step - 1) % len(ordered_fns)]

        else:
            raise ValueError("strategy must be 'random' or 'fixed'")

        current_text = attack_fn(current_text)
        new_pred, new_prob = predict_one(current_text, model, tokenizer)

        history.append({
            "step": step,
            "attack_name": attack_name,
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300]
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
sample_text = test_df.iloc[0]["text"]

result = iterative_addition_attack(
    sample_text,
    model,
    tokenizer,
    max_steps=5,
    strategy="fixed",
    seed=42,
)

print("Original prediction:", result["original_pred"])
print("New prediction:", result["new_pred"])
print("Flipped:", result["flipped"])
print("Steps:", result["steps"])
print("Original phishing prob:", result["original_prob"])
print("New phishing prob:", result["new_prob"])

pd.DataFrame(result["history"])

Original prediction: 0
New prediction: 1
Flipped: True
Steps: 3
Original phishing prob: 0.0008727670647203922
New phishing prob: 0.929248034954071


,step,attack_name,pred,phishing_prob,text_preview
0,1,benign_prefix,0,0.002964,This is a routine internal business communicat...
1,2,benign_suffix,0,0.010509,This is a routine internal business communicat...
2,3,contradiction,1,0.929248,No action is required at this time. This messa...


In [ ]:
def evaluate_iterative_addition_attack(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    strategy="random",
    seed=42,
):
    """
    Evaluates iterative addition attack over a dataframe with columns:
      - text
      - label

    Returns:
      - summary dict
      - per-example dataframe
    """

    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    for i, row in df_local.iterrows():
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
            strategy=strategy,
            seed=seed + i,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
iter_add_val_summary, iter_add_val_details = evaluate_iterative_addition_attack(
    val_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=5,
    strategy="random",
    seed=42,
)

pd.DataFrame([iter_add_val_summary])

Iterative addition eval:   0%|          | 0/300 [00:00<?, ?it/s]

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,iterative_addition_5steps,0.956667,0.9,0.980198,0.938389,0.998308,0.046667,4.896667,300


In [ ]:
iter_add_test_summary, iter_add_test_details = evaluate_iterative_addition_attack(
    test_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=5,
    strategy="random",
    seed=42,
)

pd.DataFrame([iter_add_test_summary])

Iterative addition eval:   0%|          | 0/300 [00:00<?, ?it/s]

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,iterative_addition_5steps,0.61,0.167883,0.884615,0.282209,0.888195,0.353333,4.06,300


In [ ]:
iter_add_val_summary["dataset"] = "validation"
iter_add_test_summary["dataset"] = "test"

iter_add_results_df = pd.DataFrame([
    iter_add_val_summary,
    iter_add_test_summary,
])

iter_add_results_df

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples,dataset
0,iterative_addition_5steps,0.956667,0.900000,0.980198,0.938389,0.998308,0.046667,4.896667,300,validation
1,iterative_addition_5steps,0.610000,0.167883,0.884615,0.282209,0.888195,0.353333,4.060000,300,test


In [ ]:
iter_add_test_details[iter_add_test_details["flipped"] == True].head(10)

,label,original_pred,new_pred,flipped,steps,original_prob,new_prob,original_text,attacked_text
0,0,0,1,True,3,0.000873,0.712182,Subject: Feedback sulla collaborazione con sol...,No action is required at this time. This messa...
2,0,1,0,True,1,0.507598,0.048137,Subject: Exploring Collaboration Opportunities...,This is a simulated phishing-awareness trainin...
3,0,0,1,True,1,0.037117,0.846387,Subject: Clarifications on e-commerce app enha...,No action is required at this time. This messa...
4,0,0,1,True,2,0.152537,0.801159,Subject: Opportunity for Collaboration on Digi...,This is a simulated phishing-awareness trainin...
6,0,0,1,True,3,0.000989,0.516409,Subject: Confirmation of Meeting with SmithTec...,No action is required at this time. This messa...
9,0,1,0,True,2,0.715623,0.067917,Subject: Internal Meeting: Online Accessories ...,This is a general update regarding internal do...
11,0,1,0,True,1,0.514705,0.001595,Subject: Request for Technical Specifications ...,This is a general update regarding internal do...
12,0,1,0,True,3,0.973266,0.276935,Subject: Company Performance Update and Future...,This is a general update regarding internal do...
14,0,0,1,True,5,0.026734,0.805326,Subject: Exploring the New Inventory System Fe...,No action is required at this time. This messa...
21,0,0,1,True,1,0.432693,0.617051,Subject: Request for Information on New Eco-Fr...,This is a simulated phishing-awareness trainin...


In [ ]:
def iterative_addition_attack_greedy(
    text,
    model,
    tokenizer,
    max_steps=5,
):
    attack_functions = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        best_text = None
        best_name = None
        best_pred = None
        best_prob = None

        # choose the addition that most reduces phishing probability
        for attack_name, attack_fn in attack_functions:
            candidate_text = attack_fn(current_text)
            candidate_pred, candidate_prob = predict_one(candidate_text, model, tokenizer)

            if best_prob is None or candidate_prob < best_prob:
                best_text = candidate_text
                best_name = attack_name
                best_pred = candidate_pred
                best_prob = candidate_prob

        current_text = best_text

        history.append({
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
def iterative_addition_attack_greedy_fast(
    text,
    model,
    tokenizer,
    max_steps=5,
):
    attack_functions = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one_fast(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in attack_functions:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts, model, tokenizer, batch_size=len(candidate_texts)
        )

        best_idx = int(np.argmin(candidate_probs))
        best_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = candidate_preds[best_idx]
        best_prob = candidate_probs[best_idx]

        current_text = best_text

        history.append({
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
greedy_result = iterative_addition_attack_greedy(
    test_df.iloc[0]["text"],
    model,
    tokenizer,
    max_steps=5,
)

pd.DataFrame(greedy_result["history"])

,step,attack_name,pred,phishing_prob,text_preview
0,1,noise_injection,0,0.000440,This is a general update regarding internal do...
1,2,noise_injection,0,0.000328,This is a general update regarding internal do...
2,3,noise_injection,0,0.000277,This is a general update regarding internal do...
3,4,noise_injection,0,0.000295,This is a general update regarding internal do...
4,5,noise_injection,0,0.000298,This is a general update regarding internal do...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
from tqdm.auto import tqdm

def evaluate_iterative_addition_attack(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    strategy="random",
    seed=42,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Iterative addition eval")

    for i, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
            strategy=strategy,
            seed=seed + i,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
iter_add_val_summary_full, iter_add_val_details_full = evaluate_iterative_addition_attack(
    val_df, model, tokenizer, max_samples=None, max_steps=5, strategy="random", seed=42
)

iter_add_test_summary_full, iter_add_test_details_full = evaluate_iterative_addition_attack(
    test_df, model, tokenizer, max_samples=None, max_steps=5, strategy="random", seed=42
)

pd.DataFrame([
    {"dataset": "validation", **iter_add_val_summary_full},
    {"dataset": "test", **iter_add_test_summary_full},
])

Iterative addition eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Iterative addition eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,validation,iterative_addition_5steps,0.967677,0.924948,0.985196,0.954122,0.997934,0.030556,4.932828,3960
1,test,iterative_addition_5steps,0.717875,0.691654,0.827885,0.753663,0.850767,0.289254,4.183012,11502


In [ ]:
def evaluate_iterative_addition_attack_greedy(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Greedy iterative addition eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack_greedy(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_greedy_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import pandas as pd
import numpy as np

def evaluate_iterative_addition_attack_greedy_fast(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Greedy iterative addition eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack_greedy_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_greedy_fast_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
greedy_val_summary, greedy_val_details = evaluate_iterative_addition_attack_greedy(
    val_df, model, tokenizer, max_samples=None, max_steps=5
)

greedy_test_summary, greedy_test_details = evaluate_iterative_addition_attack_greedy(
    test_df, model, tokenizer, max_samples=None, max_steps=5
)

pd.DataFrame([
    {"dataset": "validation", **greedy_val_summary},
    {"dataset": "test", **greedy_test_summary},
])

Greedy iterative addition eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Greedy iterative addition eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,validation,iterative_addition_greedy_5steps,0.973737,0.993666,0.928942,0.960214,0.998666,0.024495,4.940404,3960
1,test,iterative_addition_greedy_5steps,0.688141,0.976652,0.411608,0.579139,0.831201,0.283690,4.218397,11502


In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

os.environ["TOKENIZERS_PARALLELISM"] = "true"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def predict_one_fast(text, model, tokenizer, max_length=512):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

    probs = probs.detach().float().cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1])

def batch_predict_fast(texts, model, tokenizer, batch_size=256, max_length=512):
    preds = []
    probs_out = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        preds.extend(np.argmax(probs, axis=1).tolist())
        probs_out.extend(probs[:, 1].tolist())

    return preds, probs_out

In [ ]:
def get_token_saliency_fast(text, model, tokenizer, max_length=512):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"].to(model.device, non_blocking=True)
    attention_mask = enc["attention_mask"].to(model.device, non_blocking=True)

    embedding_layer = model.get_input_embeddings()
    inputs_embeds = embedding_layer(input_ids).detach()
    inputs_embeds.requires_grad_(True)

    model.zero_grad(set_to_none=True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    pred_class = torch.argmax(outputs.logits, dim=1)
    score = outputs.logits[0, pred_class]
    score.backward()

    grads = inputs_embeds.grad[0]
    saliency = grads.norm(dim=1).detach().float().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    return list(zip(tokens, saliency.tolist()))

In [ ]:
def merge_wordpiece_tokens(token_scores):
    words = []
    current_word = ""
    current_score = 0.0

    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}

    for tok, score in token_scores:
        if tok in special_tokens:
            continue

        if tok.startswith("##"):  # BERT
            current_word += tok[2:]
            current_score += score
        elif tok.startswith("Ġ"):  # RoBERTa
            if current_word:
                words.append((current_word, current_score))
            current_word = tok[1:]
            current_score = score
        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    cleaned = []
    for w, s in words:
        w = w.strip()
        w = re.sub(r"[^\w@.\-:/]", "", w)
        if w:
            cleaned.append((w, s))

    return cleaned

def important_word_deletion_attack_fast(text, model, tokenizer, k=1):
    token_scores = get_token_saliency_fast(text, model, tokenizer)
    word_scores = merge_wordpiece_tokens(token_scores)
    ranked_words = sorted(word_scores, key=lambda x: x[1], reverse=True)

    stop = {
        "subject", "body", "the", "a", "an", "and", "or", "to", "of", "in",
        "for", "on", "at", "is", "are", "this", "that", "with", "from"
    }

    targets = []
    for w, _ in ranked_words:
        wl = w.lower()
        if len(wl) >= 3 and wl not in stop and wl.isprintable():
            targets.append(wl)
        if len(targets) >= k:
            break

    attacked_words = []
    for w in text.split():
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in targets:
            continue
        attacked_words.append(w)

    return " ".join(attacked_words)

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "This is a routine internal business communication from the IT department. "
        "This message is safe, legitimate, and part of standard company operations.\n\n"
    )
    return prefix + text

def benign_suffix_attack(text):
    suffix = (
        "\n\nThis is a routine internal message. "
        "Please contact the internal IT help desk if you have any questions."
    )
    return text + suffix

def contradiction_attack(text):
    prefix = (
        "No action is required at this time. "
        "This message is only informational and does not require account verification.\n\n"
    )
    return prefix + text

def training_context_attack(text):
    prefix = (
        "This is a simulated phishing-awareness training email used for internal education. "
        "It is safe and authorized.\n\n"
    )
    return prefix + text

def noise_injection_attack(text, repeat=3):
    noise = (
        "This is a general update regarding internal documentation, procedures, "
        "and routine operational processes. "
    )
    return (noise * repeat) + text

In [ ]:
def hybrid_add_then_delete_attack_fast(
    text,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
):
    """
    Phase 1: greedy batched additions
    Phase 2: saliency-based deletions

    Returns:
      attacked_text, flipped, total_steps, original/new pred and prob, history
    """

    original_pred, original_prob = predict_one_fast(text, model, tokenizer, max_length=max_length)
    current_text = text
    history = []

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    # Phase 1: greedy additions, scored in one batch per step
    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts,
            model,
            tokenizer,
            batch_size=len(candidate_texts),
            max_length=max_length,
        )

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = candidate_preds[best_idx]
        best_prob = candidate_probs[best_idx]

        history.append({
            "phase": "add",
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": step,
                "steps_delete": 0,
                "total_steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    # Phase 2: saliency deletion
    for dstep in range(1, delete_steps + 1):
        new_text = important_word_deletion_attack_fast(
            current_text,
            model,
            tokenizer,
            k=1
        )

        if new_text == current_text:
            break

        current_text = new_text
        new_pred, new_prob = predict_one_fast(current_text, model, tokenizer, max_length=max_length)

        history.append({
            "phase": "delete",
            "step": dstep,
            "attack_name": "important_word_delete_k1",
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300],
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": add_steps,
                "steps_delete": dstep,
                "total_steps": add_steps + dstep,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer, max_length=max_length)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps_add": add_steps,
        "steps_delete": delete_steps,
        "total_steps": add_steps + delete_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
def evaluate_hybrid_add_then_delete_attack_fast(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    total_steps_used = []
    add_steps_used = []
    delete_steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Hybrid fast eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = hybrid_add_then_delete_attack_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            add_steps=add_steps,
            delete_steps=delete_steps,
            max_length=max_length,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])

        total_steps_used.append(result["total_steps"])
        add_steps_used.append(result["steps_add"])
        delete_steps_used.append(result["steps_delete"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps_add": result["steps_add"],
            "steps_delete": result["steps_delete"],
            "total_steps": result["total_steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_total_steps_used": float(np.mean(total_steps_used)),
        "avg_add_steps_used": float(np.mean(add_steps_used)),
        "avg_delete_steps_used": float(np.mean(delete_steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
hybrid_val_summary_fast, hybrid_val_details_fast = evaluate_hybrid_add_then_delete_attack_fast(
    val_df,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
)

hybrid_test_summary_fast, hybrid_test_details_fast = evaluate_hybrid_add_then_delete_attack_fast(
    test_df,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
)

pd.DataFrame([
    {"dataset": "validation", **hybrid_val_summary_fast},
    {"dataset": "test", **hybrid_test_summary_fast},
])

Hybrid fast eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Hybrid fast eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used,n_samples
0,validation,hybrid_fast_add3_delete5,0.972222,0.98974,0.928201,0.957983,0.998193,0.027020,7.864141,2.980051,4.884091,3960
1,test,hybrid_fast_add3_delete5,0.706660,0.97023,0.451134,0.615893,0.852527,0.295775,6.352113,2.672666,3.679447,11502


In [ ]:
hybrid_test_details_fast[hybrid_test_details_fast["flipped"] == True].head(10)